In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime

print("="*70)
print("SILVER LAYER TRANSFORMATIONS - Patient dataset")
print("="*70)


# Storage account configuration
storage_account = "jehangirdatalake05"
bronze_base_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/"
silver_base_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/"

# ============================================
# SECTION 1: SILVER LAYER TRANSFORMATIONS
# ============================================
print("\n" + "="*60)
print("SECTION 1: Reading data from bronze and writing to silver + registering tables in unity catalog's silver schema")
print("="*60)

# ============================================
# 1.1 PATIENTS TABLE - Hospital A
# ============================================
print("\n1.1 Processing Patients Data (Bronze → Silver)...")

# Read from Bronze
bronze_patients_path = f"{bronze_base_path}Hospital-A/patients"
df_patients_bronze_a = spark.read.format("parquet").load(bronze_patients_path)
df_patients_bronze_a_new = df_patients_bronze_a.withColumn("datasource",lit("hosa"))
df_patients_bronze_a_new.createOrReplaceTempView("patients_hosa")

# ============================================
# 1.1 PATIENTS TABLE - Hospital B
# ============================================

# Read from Bronze
bronze_patients_path = f"{bronze_base_path}Hospital-B/patients"
df_patients_bronze_b = spark.read.format("parquet").load(bronze_patients_path)
df_patients_bronze_b_new = df_patients_bronze_b.withColumn("datasource",lit("hosb"))
df_patients_bronze_b_new.createOrReplaceTempView("patients_hosb")

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW cdm_patients AS
SELECT CONCAT(SRC_PatientID,'-', datasource) AS Patient_Key, *
FROM (
    SELECT 
        patient_id AS SRC_PatientID ,
        first_name as FirstName,
        last_name as LastName,
        patient_age as Patient_Age,
        modified_date as ModifiedDate,
        datasource
    FROM patients_hosb
    UNION ALL
    SELECT 
        id AS SRC_PatientID,
        f_name as FirstName,
        l_name as LastName,
        age as Patient_Age,
        update_date AS ModifiedDate,
        datasource
    FROM patients_hosa);


CREATE OR REPLACE TEMP VIEW quality_checks AS
SELECT 
    Patient_Key,
    SRC_PatientID,
    FirstName,
    LastName,
    Patient_Age,
    ModifiedDate As SRC_ModifiedDate,
    datasource,
    CASE 
        WHEN SRC_PatientID IS NULL OR Patient_age IS NULL OR FirstName IS NULL THEN TRUE
        ELSE FALSE
    END AS is_quarantined
FROM cdm_patients;

CREATE TABLE if not EXISTS jehangir_catalog.silver.patients_silver (
Patient_Key STRING,
SRC_PatientID STRING,
FirstName STRING,
LastName STRING,
Patient_Age int,
SRC_ModifiedDate TIMESTAMP,
datasource STRING,
is_quarantined BOOLEAN,
inserted_date TIMESTAMP,
modified_date TIMESTAMP
)
USING DELTA;

MERGE INTO jehangir_catalog.silver.patients_silver AS target
USING quality_checks AS source
ON target.Patient_Key = source.Patient_Key
WHEN MATCHED THEN
  UPDATE SET
    target.SRC_PatientID = source.SRC_PatientID,
    target.FirstName = source.FirstName,
    target.LastName = source.LastName,
    target.Patient_Age = source.Patient_Age,
    target.SRC_ModifiedDate = source.SRC_ModifiedDate,
    target.datasource = source.datasource,
    target.is_quarantined = source.is_quarantined,
    target.modified_date = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (
    Patient_Key,
    SRC_PatientID,
    FirstName,
    LastName,
    Patient_Age,
    SRC_ModifiedDate,
    datasource,
    is_quarantined,
    inserted_date,
    modified_date
  )
  VALUES (
    source.Patient_Key,
    source.SRC_PatientID,
    source.FirstName,
    source.LastName,
    source.Patient_Age,
    source.SRC_ModifiedDate,
    source.datasource,
    source.is_quarantined,
    current_timestamp(),
    current_timestamp()
  );


In [0]:
# Optional: Write patients_silver table from Unity Catalog to Silver layer in ADLS (Data Lake)

storage_account = "jehangirdatalake05"
silver_patients_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/Hospital-A/patients"

df_patients_silver = spark.table("jehangir_catalog.silver.patients_silver")

df_patients_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(silver_patients_path)

print(f"  ✓ Successfully wrote patients_silver to: {silver_patients_path}")
print(f"  ✓ Records written: {df_patients_silver.count()}")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime
from pyspark.sql import SparkSession, functions as f

# Storage account configuration
storage_account = "jehangirdatalake05"
bronze_base_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/"
silver_base_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/"

print("Reading department data from Hospital-A")
bronze_department_path = f"{bronze_base_path}Hospital-A/departments"
df_dept_bronze_a = spark.read.format("parquet").load(bronze_department_path)
df_dept_bronze_a_new = df_dept_bronze_a.withColumn("datasource",lit("hosa"))

print("Reading department data from Hospital-B")
bronze_department_path = f"{bronze_base_path}Hospital-B/departments"
df_dept_bronze_b = spark.read.format("parquet").load(bronze_department_path)
df_dept_bronze_b_new = df_dept_bronze_b.withColumn("datasource",lit("hosb"))

df_merged = df_dept_bronze_a_new.unionByName(df_dept_bronze_b_new)


df_merged_final = df_merged.withColumn("SRC_Dept_id", f.col("department_id")) \
                     .withColumn("Dept_id", f.concat(f.col("department_id"),f.lit('-'), f.col("datasource"))) \
                     .drop("department_id")

df_merged_final.show()
df_merged_final.createOrReplaceTempView("departments")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS jehangir_catalog.silver.departments_silver (
Dept_Id string,
SRC_Dept_Id string,
dept_name string,
datasource string,
is_quarantined boolean
)
USING DELTA;

MERGE INTO jehangir_catalog.silver.departments_silver AS t
USING (
  SELECT Dept_Id, SRC_Dept_Id, dept_name, datasource, is_quarantined
  FROM (
    SELECT
      Dept_Id,
      SRC_Dept_Id,
      dept_name,
      datasource,
      CASE WHEN SRC_Dept_Id IS NULL OR dept_name IS NULL THEN TRUE ELSE FALSE END AS is_quarantined,
      row_number() OVER (PARTITION BY Dept_Id ORDER BY dept_name) AS rn
    FROM departments
  )
  WHERE rn = 1
) AS s
ON t.Dept_Id <=> s.Dept_Id
WHEN MATCHED AND NOT (
       t.SRC_Dept_Id   <=> s.SRC_Dept_Id
   AND t.dept_name     <=> s.dept_name
   AND t.datasource    <=> s.datasource
   AND t.is_quarantined <=> s.is_quarantined
) THEN UPDATE SET
  t.SRC_Dept_Id    = s.SRC_Dept_Id,
  t.dept_name      = s.dept_name,
  t.datasource     = s.datasource,
  t.is_quarantined = s.is_quarantined
WHEN NOT MATCHED THEN INSERT (Dept_Id, SRC_Dept_Id, dept_name, datasource, is_quarantined)
  VALUES (s.Dept_Id, s.SRC_Dept_Id, s.dept_name, s.datasource, s.is_quarantined);

select * from jehangir_catalog.silver.departments_silver;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime
from pyspark.sql import SparkSession, functions as f

# Storage account configuration
storage_account = "jehangirdatalake05"
bronze_base_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/"
silver_base_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/"

print("Reading providers data from Hospital-A")
bronze_providers_path = f"{bronze_base_path}Hospital-A/providers"
df_prov_bronze_a = spark.read.format("parquet").load(bronze_providers_path)
df_prov_bronze_a_new = df_prov_bronze_a.withColumn("datasource",lit("hosa"))

print("Reading providers data from Hospital-B")
bronze_providers_path = f"{bronze_base_path}Hospital-B/providers"
df_prov_bronze_b = spark.read.format("parquet").load(bronze_providers_path)
df_prov_bronze_b_new = df_prov_bronze_b.withColumn("datasource",lit("hosb"))

df_merged = df_prov_bronze_a_new.unionByName(df_prov_bronze_b_new)
df_merged.createOrReplaceTempView("providers")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS jehangir_catalog.silver.providers_silver (
provider_id string,
provider_name string,
npi_number string,
dep_id string,
datasource string,
is_quarantined boolean
)
USING DELTA;


MERGE INTO jehangir_catalog.silver.providers_silver AS t
USING (
  SELECT provider_id, provider_name, npi_number, dep_id, datasource, is_quarantined
  FROM (
    SELECT
      provider_id,
      provider_name,
      npi_number,
      dep_id,
      datasource,
      CASE WHEN provider_id IS NULL OR dep_id IS NULL THEN TRUE ELSE FALSE END AS is_quarantined,
      row_number() OVER (PARTITION BY provider_id, datasource
                         ORDER BY provider_name, npi_number, dep_id) AS rn
    FROM providers
  )
  WHERE rn = 1
) AS s
ON  t.provider_id <=> s.provider_id
AND t.datasource  <=> s.datasource
WHEN MATCHED AND NOT (
       t.provider_name  <=> s.provider_name
   AND t.npi_number     <=> s.npi_number
   AND t.dep_id         <=> s.dep_id
   AND t.is_quarantined <=> s.is_quarantined
) THEN UPDATE SET
  t.provider_name  = s.provider_name,
  t.npi_number     = s.npi_number,
  t.dep_id         = s.dep_id,
  t.is_quarantined = s.is_quarantined
WHEN NOT MATCHED THEN INSERT (provider_id, provider_name, npi_number, dep_id, datasource, is_quarantined)
  VALUES (s.provider_id, s.provider_name, s.npi_number, s.dep_id, s.datasource, s.is_quarantined);

select * from jehangir_catalog.silver.providers_silver;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime

# Storage account configuration
storage_account = "jehangirdatalake05"
bronze_base_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/"
silver_base_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/"

print("Reading encounters data from Hospital-A")
bronze_enc_path = f"{bronze_base_path}Hospital-A/encounters"
df_enc_bronze_a = spark.read.format("parquet").load(bronze_enc_path)
df_enc_bronze_a_new = df_enc_bronze_a.withColumn("datasource",lit("hosa"))

print("Reading encounters data from Hospital-B")
bronze_enc_path = f"{bronze_base_path}Hospital-B/encounters"
df_enc_bronze_b = spark.read.format("parquet").load(bronze_enc_path)
df_enc_bronze_b_new = df_enc_bronze_b.withColumn("datasource",lit("hosb"))

df_enc_bronze_a_new.createOrReplaceTempView("encounters_a")
df_enc_bronze_b_new.createOrReplaceTempView("encounters_b")

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW encounters AS
    SELECT * FROM encounters_a
    UNION ALL
    SELECT * FROM encounters_b;

CREATE OR REPLACE TEMP VIEW quality_checks AS
SELECT 
    concat(encounter_id,'-',datasource) as EncounterID,
    encounter_id as SRC_EncounterID,
    patient_id,
    provider_id,
    encounter_date,
    total_amount,
    amount_paid,
    amount_pending,
    payment_status,
    inserted_date as SRC_InsertedDate,
    modified_date as SRC_ModifiedDate,
    datasource,
    CASE 
        WHEN encounter_id IS NULL OR patient_id IS NULL THEN TRUE
    ELSE FALSE
    END AS is_quarantined
FROM encounters;


CREATE TABLE IF NOT EXISTS jehangir_catalog.silver.encounters_silver (
EncounterID string,
SRC_EncounterID string,
patient_id string,
provider_id string,
encounter_date date,
total_amount double,
amount_paid double,
amount_pending double,
payment_status string,
SRC_InsertedDate date,
SRC_ModifiedDate date,
datasource string,
is_quarantined boolean,
audit_insertdate timestamp,
audit_modifieddate timestamp
)
USING DELTA;

MERGE INTO jehangir_catalog.silver.encounters_silver AS target
USING quality_checks AS source
ON target.EncounterID = source.EncounterID
WHEN MATCHED THEN
  UPDATE SET
    target.SRC_EncounterID = source.SRC_EncounterID,
    target.patient_id = source.patient_id,
    target.provider_id = source.provider_id,
    target.encounter_date = source.encounter_date,
    target.total_amount = source.total_amount,
    target.amount_paid = source.amount_paid,
    target.amount_pending = source.amount_pending,
    target.payment_status = source.payment_status,
    target.SRC_InsertedDate = source.SRC_InsertedDate,
    target.SRC_ModifiedDate = source.SRC_ModifiedDate,
    target.datasource = source.datasource,
    target.is_quarantined = source.is_quarantined,
    target.audit_modifieddate = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (
    EncounterID,
    SRC_EncounterID,
    patient_id,
    provider_id,
    encounter_date,
    total_amount,
    amount_paid,
    amount_pending,
    payment_status,
    SRC_InsertedDate,
    SRC_ModifiedDate,
    datasource,
    is_quarantined,
    audit_insertdate,
    audit_modifieddate
  )
  VALUES (
    source.EncounterID,
    source.SRC_EncounterID,
    source.patient_id,
    source.provider_id,
    source.encounter_date,
    source.total_amount,
    source.amount_paid,
    source.amount_pending,
    source.payment_status,
    source.SRC_InsertedDate,
    source.SRC_ModifiedDate,
    source.datasource,
    source.is_quarantined,
    current_timestamp(),
    current_timestamp()
  );

  select * from jehangir_catalog.silver.encounters_silver;